# Searcher Sample: 特徴量作成サンプル

このノートブックは、`jpx-duckdb`からデータを取得し、特徴量を作成するサンプルです。

## 目的
- DuckDBから株価データを取得
- テクニカル特徴量を生成（5つ）
- シンプルなモデルで訓練・評価

## 対象ユーザー
`jpx-ls-searcher`等の別リポジトリから、このDBを利用するサーチャー（研究者）

---
## リーク防止（重要）

時系列データでは**未来の情報を使わない**ことが必須です。

### 訓練/テスト分割
```
─────────────────────────────────────────────────►時間
|<──── 訓練期間 ────>|<── テスト期間 ──>|
     2020-01-01      2023-12-31  2024-01-01    2024-12-31
                          ↑
                     ここで区切る（未来データは訓練に使わない）
```

### 特徴量とターゲット
```
 T-20  T-5  T-1   T    T+1
  │    │    │    │     │
  ▼    ▼    ▼    ▼     ▼
[過去データ] → [特徴量] → [予測対象]

・ret_1  : T-1 → T の変化率（T終値で確定 → 使用OK）
・target : T → T+1 の変化率（T+1終値が必要 → 予測対象）
```

### 注意点
- 特徴量は**当日終値時点で確定している情報**のみ使用
- ターゲット（翌日リターン）は予測対象なので特徴量に含めない
- 財務データ（EPS等）は`disclosure_date`以降のみ使用可能

---
## 1. セットアップ

In [1]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# DBパス設定（環境に応じて調整してください）
# jpx-ls-searcherから使う場合: "../../jpx-duckdb/data/jquants.duckdb"
# このリポジトリ内から使う場合: "../data/jquants.duckdb"

DB_PATH = Path("../data/jquants.duckdb")

# パス確認
print(f"DB Path: {DB_PATH.resolve()}")
print(f"Exists: {DB_PATH.exists()}")

DB Path: /Users/eleven/workspace/2026/jquants/jpx-duckdb/data/jquants.duckdb
Exists: True


---
## 2. データ取得

In [3]:
def load_price_data(
    db_path: str | Path,
    start_date: str,
    end_date: str,
    codes: list[str] | None = None
) -> pd.DataFrame:
    """
    DuckDBから株価データを取得
    
    Args:
        db_path: DuckDBファイルパス
        start_date: 開始日 (YYYY-MM-DD)
        end_date: 終了日 (YYYY-MM-DD)
        codes: 銘柄コードリスト（Noneで全銘柄）
    
    Returns:
        DataFrame: Date, Code, Open, High, Low, Close, Volume
    """
    conn = duckdb.connect(str(db_path), read_only=True)
    
    query = """
        SELECT
            b.date AS Date,
            b.code AS Code,
            b.adj_open AS Open,
            b.adj_high AS High,
            b.adj_low AS Low,
            b.adj_close AS Close,
            b.adj_volume AS Volume
        FROM eq_bars_daily b
        WHERE b.date BETWEEN ? AND ?
          AND b.adj_close IS NOT NULL
          AND b.adj_volume > 0
    """
    
    params = [start_date, end_date]
    
    if codes:
        placeholders = ", ".join(["?" for _ in codes])
        query += f" AND b.code IN ({placeholders})"
        params.extend(codes)
    
    query += " ORDER BY b.date, b.code"
    
    df = conn.execute(query, params).fetchdf()
    conn.close()
    
    return df

In [4]:
# データ取得（期間は環境に応じて調整）
df = load_price_data(
    db_path=DB_PATH,
    start_date="2020-01-01",
    end_date="2024-12-31"
)

print(f"Shape: {df.shape}")
print(f"Period: {df['Date'].min()} ~ {df['Date'].max()}")
print(f"Stocks: {df['Code'].nunique()}")
df.head()

Shape: (4968688, 7)
Period: 2020-01-06 00:00:00 ~ 2024-12-30 00:00:00
Stocks: 4846


,Date,Code,Open,High,Low,Close,Volume
0,2020-01-06,13010,2860.0,2873.0,2851.0,2862.0,25700
1,2020-01-06,13050,1781.0,1784.0,1773.0,1781.0,161540
2,2020-01-06,13060,1760.0,1767.0,1751.0,1759.0,1472020
3,2020-01-06,13080,1742.0,1748.0,1733.0,1742.0,419100
4,2020-01-06,13090,35400.0,35950.0,34800.0,35350.0,877


---
## 3. 特徴量生成

In [5]:
# 生成する特徴量リスト
FEATURES = [
    "ret_1",      # 1日リターン
    "ret_5",      # 5日リターン
    "ret_20",     # 20日リターン
    "vol_20",     # 20日ボラティリティ
    "turn_ratio", # 出来高比率（20日平均比）
]

In [6]:
def generate_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    テクニカル特徴量を生成
    
    生成する特徴量:
        - ret_1: 1日リターン (T-1 → T)
        - ret_5: 5日リターン (T-5 → T)
        - ret_20: 20日リターン (T-20 → T)
        - vol_20: 20日ボラティリティ
        - turn_ratio: 出来高比率（20日平均比）
    
    ターゲット:
        - target: 翌日リターン (T → T+1)
    
    Args:
        df: Date, Code, Open, High, Low, Close, Volume を含むDataFrame
    
    Returns:
        DataFrame: 特徴量とターゲットを追加したDataFrame
    """
    df = df.copy()
    df = df.sort_values(["Code", "Date"]).reset_index(drop=True)
    
    grp = df.groupby("Code")
    
    # === 特徴量（当日終値時点で確定） ===
    
    # リターン系
    df["ret_1"] = grp["Close"].pct_change(1)    # 1日リターン
    df["ret_5"] = grp["Close"].pct_change(5)    # 5日リターン
    df["ret_20"] = grp["Close"].pct_change(20)  # 20日リターン
    
    # ボラティリティ（20日間の日次リターンの標準偏差）
    df["vol_20"] = grp["ret_1"].transform(
        lambda x: x.rolling(20, min_periods=20).std()
    )
    
    # 出来高比率（当日出来高 / 20日平均出来高）
    df["vol_ma_20"] = grp["Volume"].transform(
        lambda x: x.rolling(20, min_periods=20).mean()
    )
    df["turn_ratio"] = df["Volume"] / df["vol_ma_20"]
    df = df.drop(columns=["vol_ma_20"])
    
    # === ターゲット（翌日リターン） ===
    # shift(-1) で翌日の終値を持ってくる
    df["target"] = grp["Close"].shift(-1) / df["Close"] - 1
    
    return df

In [7]:
# 特徴量生成
df = generate_features(df)

print(f"Shape: {df.shape}")
print(f"\nFeatures:")
for f in FEATURES:
    print(f"  {f}: {df[f].notna().sum():,} valid rows")

df[["Date", "Code", "Close"] + FEATURES + ["target"]].head(30)

Shape: (4968688, 13)

Features:
  ret_1: 4,963,842 valid rows
  ret_5: 4,944,950 valid rows
  ret_20: 4,874,471 valid rows
  vol_20: 4,874,471 valid rows
  turn_ratio: 4,879,159 valid rows


,Date,Code,Close,ret_1,ret_5,ret_20,vol_20,turn_ratio,target
0,2020-01-06,13010,2862.0,NaN,NaN,NaN,NaN,NaN,0.012579
1,2020-01-07,13010,2898.0,0.012579,NaN,NaN,NaN,NaN,0.001725
2,2020-01-08,13010,2903.0,0.001725,NaN,NaN,NaN,NaN,0.010334
3,2020-01-09,13010,2933.0,0.010334,NaN,NaN,NaN,NaN,-0.000341
4,2020-01-10,13010,2932.0,-0.000341,NaN,NaN,NaN,NaN,-0.006480
5,2020-01-14,13010,2913.0,-0.006480,0.017820,NaN,NaN,NaN,0.007552
6,2020-01-15,13010,2935.0,0.007552,0.012767,NaN,NaN,NaN,0.000341
7,2020-01-16,13010,2936.0,0.000341,0.011368,NaN,NaN,NaN,0.000341
8,2020-01-17,13010,2937.0,0.000341,0.001364,NaN,NaN,NaN,0.001702
9,2020-01-20,13010,2942.0,0.001702,0.003411,NaN,NaN,NaN,0.000340


---
## 4. 訓練/テスト分割

**重要**: 時系列データでは時間で分割します（ランダム分割はNG）

In [8]:
# 分割日（この日以前 = 訓練、この日より後 = テスト）
TRAIN_END = "2023-12-31"

# 欠損値を除去（特徴量計算に必要な過去データがない行）
df_clean = df.dropna(subset=FEATURES + ["target"])

# 時間で分割
train_df = df_clean[df_clean["Date"] <= TRAIN_END]
test_df = df_clean[df_clean["Date"] > TRAIN_END]

print(f"Train: {train_df['Date'].min()} ~ {train_df['Date'].max()} ({len(train_df):,} rows)")
print(f"Test:  {test_df['Date'].min()} ~ {test_df['Date'].max()} ({len(test_df):,} rows)")

Train: 2020-02-04 00:00:00 ~ 2023-12-29 00:00:00 (3,846,005 rows)
Test:  2024-01-04 00:00:00 ~ 2024-12-27 00:00:00 (1,023,780 rows)


In [9]:
# 特徴量とターゲットを分離
X_train = train_df[FEATURES]
y_train = train_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (3846005, 5)
X_test:  (1023780, 5)


---
## 5. モデル訓練

In [10]:
# Ridge回帰（シンプルなベースライン）
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

print("Model trained!")
print(f"\nFeature coefficients:")
for name, coef in zip(FEATURES, model.coef_):
    print(f"  {name}: {coef:.6f}")

Model trained!

Feature coefficients:
  ret_1: -0.003070
  ret_5: -0.004832
  ret_20: -0.000782
  vol_20: 0.033201
  turn_ratio: -0.000135


---
## 6. 評価

In [11]:
# 予測
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# 評価指標
print("=== Train ===")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.6f}")
print(f"  R2:   {r2_score(y_train, y_pred_train):.6f}")

print("\n=== Test ===")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.6f}")
print(f"  R2:   {r2_score(y_test, y_pred_test):.6f}")

=== Train ===
  RMSE: 0.026473
  R2:   0.000586

=== Test ===
  RMSE: 0.026220
  R2:   0.001406


In [12]:
# 予測値の分布確認
test_df = test_df.copy()
test_df["pred"] = y_pred_test

print("Prediction distribution:")
print(test_df["pred"].describe())

print("\nActual target distribution:")
print(test_df["target"].describe())

Prediction distribution:
count    1.023780e+06
mean     3.621525e-04
std      6.818511e-04
min     -1.141893e-01
25%     -8.574209e-06
50%      2.222980e-04
75%      5.725985e-04
max      4.034178e-02
Name: pred, dtype: float64

Actual target distribution:
count    1.023780e+06
mean     3.945396e-04
std      2.623860e-02
min     -8.329519e-01
25%     -8.746356e-03
50%      0.000000e+00
75%      8.849558e-03
max      1.153846e+00
Name: target, dtype: float64


---
## 7. 次のステップ

このサンプルをベースに拡張できる項目:

1. **特徴量の追加**
   - セクター特徴量（`eq_master`の`sector_17_code`を使用）
   - 財務特徴量（`fin_summary`のEPS, BPSを使用）
   - マーケット特徴量（`idx_bars_daily_topix`を使用）

2. **モデルの改善**
   - LightGBM Regressor
   - LightGBM Ranker（ランキング学習）

3. **評価の強化**
   - 日次のIC（Information Coefficient）
   - ポートフォリオリターンのバックテスト

詳細は `docs/transfer_package_v1.md` を参照してください。